# 🚀 Group B — Notebook 3B: Scale the Custom Transformer on Perlmutter

Notebook 2 taught the architecture.

Notebook 3B keeps the same readable PyTorch structure and makes it larger:

```mermaid
flowchart LR
    A["Notebook 2<br/>small teaching Transformer"] --> B["Notebook 3B"]
    B --> C["d_model = 256"]
    C --> D["8 attention heads"]
    D --> E["8 Transformer layers"]
    E --> F["Multiple GPUs with DDP"]
    F --> G["Train all 5 tokenizers"]
    G --> H["Compare biology + compute"]
```

# 1. How several GPUs train one tokenizer

```mermaid
flowchart TD
    A["One tokenizer<br/>for example overlap 6-mer"] --> B["One custom Transformer"]
    B --> C["GPU 0 / rank 0"]
    B --> D["GPU 1 / rank 1"]
    B --> E["GPU 2 / rank 2"]
    B --> F["GPU 3 / rank 3"]

    C --> G["Synchronize gradients"]
    D --> G
    E --> G
    F --> G

    G --> H["Finish this tokenizer"]
    H --> I["Move to next tokenizer"]
```

The GPUs cooperate on **one model at a time**. They are not running five independent tokenizer experiments simultaneously.

In [ ]:
# ▶️ RUN — student settings

from pathlib import Path
import json
import os
import sys
import time
import shlex
import subprocess
import py_compile
import math
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

PROJECT_DIR = Path(
    os.environ.get("DNA_BOOTCAMP_HOME", ".")
).expanduser().resolve()

DATA_DIR = Path("/global/cfs/cdirs/m4388/projects/project7/ctcf_k562_example")
RESULTS_DIR = PROJECT_DIR / "notebook3b_results"
SCRIPTS_DIR = PROJECT_DIR / "notebook3b_scripts"
LOG_DIR = RESULTS_DIR / "slurm_logs"

for folder in [RESULTS_DIR, SCRIPTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

NERSC_ACCOUNT = "m4388"

RUN_MODE = "bootcamp"
BOOTCAMP_DAY = 3

GPU_COUNT = 4
LOCAL_BATCH_SIZE = 32

MODEL_CONFIG = {
    "d_model": 256,
    "nhead": 8,
    "layers": 8,
    "dropout": 0.10,
    "epochs": 10,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "bpe_merges": 80,
    "seed": 42,
}

print("Bootcamp day:", BOOTCAMP_DAY)
print("GPUs:", GPU_COUNT)
print("Local batch per GPU:", LOCAL_BATCH_SIZE)
print("Transformer width:", MODEL_CONFIG["d_model"])
print("Transformer layers:", MODEL_CONFIG["layers"])

# 2. What changed from Notebook 2?

The architecture did **not** change:

```mermaid
flowchart LR
    A["Tokenized DNA"] --> B["Input vectors"]
    B --> C["+ Position"]
    C --> D["nn.TransformerEncoder"]
    D --> E["Sequence summary"]
    E --> F["Classifier"]
```

Only the size and training resources increased.

### Student-visible model settings

- `d_model=256` → each token representation has 256 numbers.
- `nhead=8` → eight attention heads.
- `layers=8` → eight Transformer encoder blocks.
- `dropout=0.10` → regularization.
- `epochs=10` → ten training passes.
- `LOCAL_BATCH_SIZE=32` → each GPU handles 32 examples at once.
- `learning_rate=3e-4` → optimizer update size.

In [ ]:
# 🔒 HELPER — select a Python environment that contains PyTorch

PYTHON_CANDIDATES = [
    Path("/global/cfs/cdirs/m4388/envs/dna-llm/bin/python"),
    Path.home() / ".conda" / "envs" / "dna-llm" / "bin" / "python",
    Path(sys.executable),
]

NOTEBOOK_PYTHON = None

for candidate in PYTHON_CANDIDATES:
    if not candidate.exists():
        continue

    check = subprocess.run(
        [
            str(candidate),
            "-c",
            "import torch, sklearn, numpy; print(torch.__version__)",
        ],
        capture_output=True,
        text=True,
    )

    if check.returncode == 0:
        NOTEBOOK_PYTHON = str(candidate)
        break

if NOTEBOOK_PYTHON is None:
    raise RuntimeError(
        "Could not find a Python environment with PyTorch."
    )

print("Training Python:", NOTEBOOK_PYTHON)

In [ ]:
# 🔒 HELPER — bootcamp reservation / Slurm resource plan

GPUS_PER_NODE = 4

BOOTCAMP_RESERVATIONS = {
    1: ("bootcamp_day1", 20, "11:00-22:00"),
    2: ("bootcamp_day2", 30, "11:00-22:00"),
    3: ("bootcamp_day3", 30, "11:00-22:00"),
    4: ("bootcamp_day4", 40, "08:00-00:00"),
    5: ("bootcamp_day5", 30, "08:00-11:00"),
}

def resolve_slurm(gpu_count):
    if RUN_MODE == "shared":
        if gpu_count > 2:
            raise ValueError("Shared mode supports 1-2 GPUs here.")
        return {
            "qos":"shared",
            "reservation":None,
            "nodes":1,
            "tasks_per_node":gpu_count,
            "gpus_per_node":gpu_count,
        }

    name,max_nodes,_ = BOOTCAMP_RESERVATIONS[BOOTCAMP_DAY]
    nodes=math.ceil(gpu_count/GPUS_PER_NODE)

    if nodes > 1 and gpu_count % GPUS_PER_NODE != 0:
        raise ValueError("Multi-node bootcamp jobs must use whole 4-GPU nodes.")

    if nodes > max_nodes:
        raise ValueError("GPU request is larger than the reservation.")

    per_node=gpu_count if nodes==1 else GPUS_PER_NODE

    return {
        "qos":"regular",
        "reservation":name,
        "nodes":nodes,
        "tasks_per_node":per_node,
        "gpus_per_node":per_node,
    }

In [ ]:
%%writefile notebook3b_scripts/train_all_tokenizers.py

import argparse
import itertools
import json
import os
import random
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score


def setup_ddp():
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    if world_size > 1:
        dist.init_process_group(
            backend="nccl",
            init_method="env://",
            rank=rank,
            world_size=world_size,
            device_id=device,
        )

    return rank, local_rank, world_size, device


def cleanup_ddp():
    if dist.is_initialized():
        dist.destroy_process_group()


def load_dataset(data_dir):
    data_dir = Path(data_dir)

    sequences = [
        line.strip().upper()
        for line in (data_dir / "seqs.txt").read_text().splitlines()
        if line.strip()
    ]
    labels = [
        int(line.strip())
        for line in (data_dir / "labels.txt").read_text().splitlines()
        if line.strip()
    ]

    label_sets = {}

    for seq, label in zip(sequences, labels):
        if set(seq) <= set("ACGT"):
            label_sets.setdefault(seq, set()).add(label)

    clean_sequences = [
        seq
        for seq, values in label_sets.items()
        if len(values) == 1
    ]

    clean_labels = [
        next(iter(label_sets[seq]))
        for seq in clean_sequences
    ]

    return clean_sequences, clean_labels


BASE_ID = {"A":1, "C":2, "G":3, "T":4}
ONE_HOT = {
    "A":[1.,0.,0.,0.],
    "C":[0.,1.,0.,0.],
    "G":[0.,0.,1.,0.],
    "T":[0.,0.,0.,1.],
}

ALL_6MERS = [
    "".join(chars)
    for chars in itertools.product("ACGT", repeat=6)
]
KMER_ID = {
    token:i+1
    for i,token in enumerate(ALL_6MERS)
}


class SimpleBPE:
    def __init__(self, merges=80):
        self.merges = merges
        self.rules = []
        self.vocab = {"<PAD>":0, "<UNK>":1}

    def _apply(self, tokens, pair, merged):
        out=[]
        i=0
        while i < len(tokens):
            if i < len(tokens)-1 and (tokens[i],tokens[i+1]) == pair:
                out.append(merged)
                i += 2
            else:
                out.append(tokens[i])
                i += 1
        return out

    def fit(self, sequences):
        tokenized=[list(seq) for seq in sequences]
        for _ in range(self.merges):
            counts=Counter()
            for tokens in tokenized:
                counts.update(zip(tokens[:-1],tokens[1:]))
            if not counts:
                break
            pair,_=counts.most_common(1)[0]
            merged=pair[0]+pair[1]
            self.rules.append((pair,merged))
            tokenized=[
                self._apply(tokens,pair,merged)
                for tokens in tokenized
            ]

        learned=sorted({
            token
            for tokens in tokenized
            for token in tokens
        })
        self.vocab.update({
            token:i+2
            for i,token in enumerate(learned)
        })

    def tokens(self, sequence):
        tokens=list(sequence)
        for pair,merged in self.rules:
            tokens=self._apply(tokens,pair,merged)
        return tokens

    def encode(self, sequence):
        return [
            self.vocab.get(token,1)
            for token in self.tokens(sequence)
        ]


@dataclass
class Spec:
    name: str
    input_kind: str
    vocab_size: int
    max_length: int
    encode: object


def build_specs(train_sequences, bpe_merges):
    bpe=SimpleBPE(bpe_merges)
    bpe.fit(train_sequences)

    specs = {
        "single_nucleotide":Spec(
            "single_nucleotide","token_ids",5,200,
            lambda seq:[BASE_ID[b] for b in seq],
        ),
        "one_hot":Spec(
            "one_hot","one_hot",4,200,
            lambda seq:torch.tensor(
                [ONE_HOT[b] for b in seq],
                dtype=torch.float32,
            ),
        ),
        "overlap_6mer":Spec(
            "overlap_6mer","token_ids",4097,195,
            lambda seq:[
                KMER_ID[seq[i:i+6]]
                for i in range(len(seq)-5)
            ],
        ),
        "nonoverlap_6mer":Spec(
            "nonoverlap_6mer","token_ids",4097,33,
            lambda seq:[
                KMER_ID[seq[i:i+6]]
                for i in range(0,len(seq)-5,6)
            ],
        ),
        "bpe":Spec(
            "bpe","token_ids",len(bpe.vocab),200,bpe.encode
        ),
    }

    return specs,bpe


class EncodedDataset(Dataset):
    def __init__(self, sequences, labels, spec):
        self.data=[]
        for seq,y in zip(sequences,labels):
            x=spec.encode(seq)
            if not torch.is_tensor(x):
                x=torch.tensor(x,dtype=torch.long)
            self.data.append((x,int(y)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self,index):
        return self.data[index]


def collate_batch(batch):
    xs,ys=zip(*batch)
    lengths=torch.tensor([len(x) for x in xs],dtype=torch.long)

    padding_value=0.0 if xs[0].dtype.is_floating_point else 0
    x=pad_sequence(
        xs,
        batch_first=True,
        padding_value=padding_value,
    )

    positions=torch.arange(x.shape[1]).unsqueeze(0)
    padding_mask=positions >= lengths.unsqueeze(1)

    return x,padding_mask,torch.tensor(ys,dtype=torch.long)


class DNAClassifier(nn.Module):
    def __init__(self,input_kind,vocab_size,max_length,
                 d_model,nhead,layers,dropout):
        super().__init__()

        if input_kind == "one_hot":
            self.input_layer=nn.Linear(4,d_model)
        else:
            self.input_layer=nn.Embedding(
                vocab_size,d_model,padding_idx=0
            )

        self.position=nn.Embedding(max_length,d_model)

        layer=nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True,
        )

        self.transformer=nn.TransformerEncoder(
            layer,
            num_layers=layers,
        )

        self.classifier=nn.Linear(d_model,2)

    def forward(self,x,padding_mask):
        x=self.input_layer(x)

        positions=torch.arange(
            x.shape[1],
            device=x.device,
        )
        x=x+self.position(positions)

        x=self.transformer(
            x,
            src_key_padding_mask=padding_mask,
        )

        keep=(~padding_mask).unsqueeze(-1)
        summary=(x*keep).sum(1)/keep.sum(1)

        return self.classifier(summary)


def train_one(spec, train_seq, val_seq, train_y, val_y,
              args, rank, world_size, device, bpe=None):

    train_data=EncodedDataset(train_seq,train_y,spec)
    val_data=EncodedDataset(val_seq,val_y,spec)

    sampler=DistributedSampler(
        train_data,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=args.seed,
    ) if world_size > 1 else None

    local_batch=args.local_batch_size

    train_loader=DataLoader(
        train_data,
        batch_size=local_batch,
        shuffle=(sampler is None),
        sampler=sampler,
        collate_fn=collate_batch,
        pin_memory=True,
    )

    val_loader=DataLoader(
        val_data,
        batch_size=local_batch,
        shuffle=False,
        collate_fn=collate_batch,
        pin_memory=True,
    )

    model=DNAClassifier(
        spec.input_kind,
        spec.vocab_size,
        spec.max_length,
        args.d_model,
        args.nhead,
        args.layers,
        args.dropout,
    ).to(device)

    if world_size > 1:
        model=DDP(
            model,
            device_ids=[device.index],
            output_device=device.index,
            gradient_as_bucket_view=True,
        )

    optimizer=torch.optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay,
    )
    loss_fn=nn.CrossEntropyLoss()

    torch.cuda.reset_peak_memory_stats(device)
    start=time.time()
    history=[]

    for epoch in range(1,args.epochs+1):
        if sampler is not None:
            sampler.set_epoch(epoch)

        model.train()
        total_loss=0.0
        count=0

        for x,mask,y in train_loader:
            x=x.to(device,non_blocking=True)
            mask=mask.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                logits=model(x,mask)
                loss=loss_fn(logits,y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()*len(y)
            count += len(y)

        stats=torch.tensor(
            [total_loss,float(count)],
            device=device,
            dtype=torch.float64,
        )
        if world_size > 1:
            dist.all_reduce(stats)

        train_loss=(stats[0]/stats[1]).item()

        if world_size > 1:
            dist.barrier()

        if rank == 0:
            eval_model=model.module if world_size > 1 else model
            eval_model.eval()

            truth=[]
            pred=[]
            prob=[]

            with torch.no_grad():
                for x,mask,y in val_loader:
                    x=x.to(device)
                    mask=mask.to(device)

                    with torch.amp.autocast(
                        device_type="cuda",
                        dtype=torch.bfloat16,
                    ):
                        logits=eval_model(x,mask)

                    p=torch.softmax(logits.float(),dim=1)[:,1]
                    yhat=logits.argmax(dim=1)

                    truth.extend(y.tolist())
                    pred.extend(yhat.cpu().tolist())
                    prob.extend(p.cpu().tolist())

            auroc=roc_auc_score(truth,prob)
            auprc=average_precision_score(truth,prob)

            history.append({
                "epoch":epoch,
                "train_loss":train_loss,
                "val_auroc":auroc,
                "val_auprc":auprc,
            })

            print(
                f"{spec.name:20s} | epoch {epoch}/{args.epochs} | "
                f"loss={train_loss:.4f} | AUROC={auroc:.4f}",
                flush=True,
            )

        if world_size > 1:
            dist.barrier()

    torch.cuda.synchronize(device)
    elapsed=time.time()-start

    peak=torch.tensor(
        [torch.cuda.max_memory_allocated(device)],
        device=device,
        dtype=torch.float64,
    )
    if world_size > 1:
        dist.all_reduce(peak,op=dist.ReduceOp.MAX)

    if rank == 0:
        eval_model = model.module if world_size > 1 else model

        checkpoint_path = (
            Path(args.output_dir)
            / f"{spec.name}_final_checkpoint.pt"
        )

        checkpoint = {
            "family": "custom_transformer",
            "tokenizer": spec.name,
            "state_dict": eval_model.state_dict(),
            "input_kind": spec.input_kind,
            "vocab_size": int(spec.vocab_size),
            "max_length": int(spec.max_length),
            "d_model": int(args.d_model),
            "nhead": int(args.nhead),
            "layers": int(args.layers),
            "dropout": float(args.dropout),
            "bpe_merges": int(args.bpe_merges),
        }

        # BPE learned its tokenizer from the training split.
        # Save those learned rules so Notebook 4 can tokenize
        # new DNA exactly the same way.
        if spec.name == "bpe":
            checkpoint["bpe_rules"] = bpe.rules
            checkpoint["bpe_vocab"] = bpe.vocab

        torch.save(
            checkpoint,
            checkpoint_path,
        )

        metrics={
            "tokenizer":spec.name,
            "num_gpus":world_size,
            "local_batch_size":local_batch,
            "global_batch_size":local_batch*world_size,
            "best_val_auroc":max(row["val_auroc"] for row in history),
            "final_val_auroc":history[-1]["val_auroc"],
            "final_val_auprc":history[-1]["val_auprc"],
            "final_val_accuracy":accuracy_score(truth,pred),
            "training_time_seconds":elapsed,
            "peak_gpu_memory_gb":peak.item()/(1024**3),
            "parameters":sum(
                p.numel()
                for p in eval_model.parameters()
            ),
            "checkpoint_path":str(checkpoint_path),
        }

        predictions = {
            "true": truth,
            "predicted": pred,
            "probability": prob,
        }

        print(
            f"Saved checkpoint: {checkpoint_path}",
            flush=True,
        )

        return metrics,history,predictions

    return None,None,None


def main():
    parser=argparse.ArgumentParser()

    parser.add_argument("--data_dir",required=True)
    parser.add_argument("--output_dir",required=True)

    parser.add_argument("--d_model",type=int,default=256)
    parser.add_argument("--nhead",type=int,default=8)
    parser.add_argument("--layers",type=int,default=8)
    parser.add_argument("--dropout",type=float,default=0.1)

    parser.add_argument("--epochs",type=int,default=10)
    parser.add_argument("--local_batch_size",type=int,default=32)
    parser.add_argument("--learning_rate",type=float,default=3e-4)
    parser.add_argument("--weight_decay",type=float,default=0.01)
    parser.add_argument("--bpe_merges",type=int,default=80)
    parser.add_argument("--seed",type=int,default=42)

    args=parser.parse_args()

    rank,local_rank,world_size,device=setup_ddp()

    try:
        random.seed(args.seed)
        np.random.seed(args.seed)
        torch.manual_seed(args.seed)

        sequences,labels=load_dataset(args.data_dir)

        train_seq,val_seq,train_y,val_y=train_test_split(
            sequences,
            labels,
            test_size=0.20,
            random_state=args.seed,
            stratify=labels,
        )

        specs,bpe=build_specs(train_seq,args.bpe_merges)

        output_dir=Path(args.output_dir)
        output_dir.mkdir(parents=True,exist_ok=True)

        order=[
            "single_nucleotide",
            "one_hot",
            "overlap_6mer",
            "nonoverlap_6mer",
            "bpe",
        ]

        all_metrics=[]

        if rank == 0:
            print(
                f"DDP world size: {world_size} | "
                f"model: d_model={args.d_model}, "
                f"heads={args.nhead}, layers={args.layers}",
                flush=True,
            )

        for name in order:
            metrics,history,predictions=train_one(
                specs[name],
                train_seq,val_seq,
                train_y,val_y,
                args,rank,world_size,device,
                bpe=(bpe if name == "bpe" else None),
            )

            if rank == 0:
                all_metrics.append(metrics)

                with open(
                    output_dir/f"{name}_summary.json",
                    "w",
                ) as handle:
                    json.dump(metrics,handle,indent=2)

                with open(
                    output_dir/f"{name}_history.json",
                    "w",
                ) as handle:
                    json.dump(history,handle,indent=2)

                with open(
                    output_dir/f"{name}_predictions.json",
                    "w",
                ) as handle:
                    json.dump(predictions,handle,indent=2)

            if world_size > 1:
                dist.barrier()

        if rank == 0:
            with open(
                output_dir/"all_tokenizers_summary.json",
                "w",
            ) as handle:
                json.dump(all_metrics,handle,indent=2)

            print("Saved combined results.",flush=True)

    finally:
        cleanup_ddp()


if __name__ == "__main__":
    main()

In [ ]:
# 🔒 HELPER — create/submit/monitor the Slurm job

TRAIN_SCRIPT = SCRIPTS_DIR / "train_all_tokenizers.py"
py_compile.compile(str(TRAIN_SCRIPT), doraise=True)


def build_job():
    plan=resolve_slurm(GPU_COUNT)

    job_name="nb3b-transformers"
    slurm_path=SCRIPTS_DIR/"notebook3b.slurm"

    header=[
        "#!/bin/bash",
        f"#SBATCH -A {NERSC_ACCOUNT}",
        "#SBATCH -C gpu",
        f"#SBATCH -q {plan['qos']}",
        "#SBATCH -t 01:30:00",
        f"#SBATCH -N {plan['nodes']}",
        f"#SBATCH --ntasks-per-node={plan['tasks_per_node']}",
        "#SBATCH --cpus-per-task=32",
        f"#SBATCH --gpus-per-node={plan['gpus_per_node']}",
        "#SBATCH --gpu-bind=none",
        f"#SBATCH -J {job_name}",
        f"#SBATCH -o {LOG_DIR}/{job_name}-%j.out",
    ]

    if plan["reservation"]:
        header.insert(
            4,
            f"#SBATCH --reservation={plan['reservation']}",
        )

    args=[
        "--data_dir",str(DATA_DIR),
        "--output_dir",str(RESULTS_DIR),
        "--d_model",str(MODEL_CONFIG["d_model"]),
        "--nhead",str(MODEL_CONFIG["nhead"]),
        "--layers",str(MODEL_CONFIG["layers"]),
        "--dropout",str(MODEL_CONFIG["dropout"]),
        "--epochs",str(MODEL_CONFIG["epochs"]),
        "--local_batch_size",str(LOCAL_BATCH_SIZE),
        "--learning_rate",str(MODEL_CONFIG["learning_rate"]),
        "--weight_decay",str(MODEL_CONFIG["weight_decay"]),
        "--bpe_merges",str(MODEL_CONFIG["bpe_merges"]),
        "--seed",str(MODEL_CONFIG["seed"]),
    ]

    arg_text=" ".join(shlex.quote(x) for x in args)

    body=f'''
export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT=$((10000 + SLURM_JOB_ID % 50000))
export NCCL_DEBUG=VERSION

unset CUDA_VISIBLE_DEVICES
unset NVIDIA_VISIBLE_DEVICES
unset ROCR_VISIBLE_DEVICES
unset GPU_DEVICE_ORDINAL

srun --gpu-bind=none bash -c '
    export RANK=$SLURM_PROCID
    export LOCAL_RANK=$SLURM_LOCALID
    export WORLD_SIZE=$SLURM_NTASKS
    {NOTEBOOK_PYTHON} {TRAIN_SCRIPT} {arg_text}
'
'''

    slurm_path.write_text("\n".join(header)+"\n"+body)

    check=subprocess.run(
        ["bash","-n",str(slurm_path)],
        capture_output=True,
        text=True,
    )
    if check.returncode != 0:
        raise RuntimeError(check.stderr)

    return slurm_path,job_name


def submit_and_stream(script_path, job_name):
    env=os.environ.copy()

    for name in [
        "CUDA_VISIBLE_DEVICES",
        "NVIDIA_VISIBLE_DEVICES",
        "ROCR_VISIBLE_DEVICES",
        "GPU_DEVICE_ORDINAL",
    ]:
        env.pop(name,None)

    submit=subprocess.run(
        ["sbatch",str(script_path)],
        capture_output=True,
        text=True,
        env=env,
        cwd=PROJECT_DIR,
    )

    if submit.returncode != 0:
        raise RuntimeError(submit.stderr)

    job_id=submit.stdout.strip().split()[-1]
    log_path=LOG_DIR/f"{job_name}-{job_id}.out"

    print("Submitted job",job_id)

    last_size=0

    while True:
        if log_path.exists():
            with log_path.open() as handle:
                handle.seek(last_size)
                text=handle.read()
                if text:
                    print(text,end="")
                last_size=handle.tell()

        active=subprocess.run(
            ["squeue","-h","-j",job_id],
            capture_output=True,
            text=True,
        ).stdout.strip()

        if not active:
            break

        time.sleep(2)

    if log_path.exists():
        with log_path.open() as handle:
            handle.seek(last_size)
            text=handle.read()
            if text:
                print(text,end="")

    state=subprocess.run(
        ["sacct","-j",job_id,"--format=JobID,State,ExitCode","-n","-P"],
        capture_output=True,
        text=True,
    ).stdout

    main=[
        line for line in state.splitlines()
        if line.split("|")[0] == job_id
    ]

    if not main or not main[0].split("|")[1].startswith("COMPLETED"):
        raise RuntimeError(f"Job {job_id} did not complete successfully.")

    return job_id

# 3. From settings to a Slurm job

```mermaid
flowchart LR
    A["Student settings"] --> B["build_job()"]
    B --> C["Slurm script"]
    C --> D["submit_and_stream(...)"]
    D --> E["Perlmutter GPUs"]
    E --> F["DDP trains tokenizer 1"]
    F --> G["Tokenizer 2 ... tokenizer 5"]
    G --> H["Saved summaries + predictions"]
```

## Function: `build_job()`

Converts the visible settings into the Slurm script.

## Function: `submit_and_stream(...)`

Submits that script and displays the important training output.

# 3. Prepare the job

The hidden helpers take your simple settings and turn them into a Slurm script.

You do not need to write `#SBATCH` directives yourself.

In [ ]:
# ▶️ RUN

SLURM_FILE, JOB_NAME = build_job()

print("Job ready.")
print("Slurm file:", SLURM_FILE)
print("GPUs:", GPU_COUNT)
print("Tokenizers: 5")

# 4. Train all five tokenizers

This one call sends the work to Perlmutter.

For each tokenizer the log will show something like:

```text
overlap_6mer | epoch 1/10 | loss=... | AUROC=...
```

Then it moves to the next representation.

In [ ]:
# ▶️ RUN

JOB_ID = submit_and_stream(
    SLURM_FILE,
    JOB_NAME,
)

## The trained models are now saved too

Notebook 3B now saves one reusable checkpoint for every tokenizer:

```text
notebook3b_results/
├── single_nucleotide_final_checkpoint.pt
├── one_hot_final_checkpoint.pt
├── overlap_6mer_final_checkpoint.pt
├── nonoverlap_6mer_final_checkpoint.pt
└── bpe_final_checkpoint.pt
```

A checkpoint contains:

```text
model architecture settings
+
learned model weights
+
tokenizer information
```

The BPE checkpoint additionally stores the **BPE rules and vocabulary learned from the training DNA**.

Notebook 4 uses these files to make predictions on new ENCSR000AOO astrocyte DNA.

# 5. Compare the results

The long training program saves one small combined JSON file.

We only need that file for the final comparison.

In [ ]:
# 🔒 HELPER — load combined results and predictions for the best tokenizer

SUMMARY_PATH = RESULTS_DIR / "all_tokenizers_summary.json"

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "Combined results were not found. Check that the Slurm job completed."
    )

with open(SUMMARY_PATH) as handle:
    all_results = json.load(handle)

comparison = pd.DataFrame(
    all_results
)

best_row = comparison.loc[
    comparison["best_val_auroc"].idxmax()
]

BEST_TOKENIZER = best_row["tokenizer"]

PREDICTIONS_PATH = (
    RESULTS_DIR
    / f"{BEST_TOKENIZER}_predictions.json"
)

HISTORY_PATH = (
    RESULTS_DIR
    / f"{BEST_TOKENIZER}_history.json"
)

with open(PREDICTIONS_PATH) as handle:
    best_predictions = json.load(handle)

with open(HISTORY_PATH) as handle:
    best_history = pd.DataFrame(
        json.load(handle)
    )

In [ ]:
# ▶️ RUN — pandas gives us one comparison table

comparison[
    [
        "tokenizer",
        "best_val_auroc",
        "final_val_auprc",
        "final_val_accuracy",
        "training_time_seconds",
        "peak_gpu_memory_gb",
        "parameters",
    ]
].round(3)

In [ ]:
# ▶️ RUN — AUROC across tokenizers

comparison.plot(
    x="tokenizer",
    y="best_val_auroc",
    kind="bar",
    legend=False,
)

plt.ylim(0, 1)
plt.ylabel("Best validation AUROC")
plt.title("Which tokenizer performed best?")
plt.xticks(rotation=30, ha="right")
plt.show()

In [ ]:
# ▶️ RUN — training time across tokenizers

comparison.plot(
    x="tokenizer",
    y="training_time_seconds",
    kind="bar",
    legend=False,
)

plt.ylabel("Training time (seconds)")
plt.title("Which tokenizer cost more computation?")
plt.xticks(rotation=30, ha="right")
plt.show()

# 6. Look closely at the best tokenizer model

The comparison graphs tell us **which run looked strongest overall**.

Now we inspect the best-AUROC tokenizer with the same model-evaluation graphs used in Group A.

This is important: even if Group A and Group B use different model architectures, they should evaluate their classifiers with the same language.

In [ ]:
print("Best tokenizer by AUROC:", BEST_TOKENIZER)

best_row.to_frame(
    name="value"
)

## Learning curve

This shows whether the custom Transformer continued improving across epochs.

In [ ]:
plt.plot(
    best_history["epoch"],
    best_history["train_loss"],
    marker="o",
    label="Training loss",
)

plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title(f"{BEST_TOKENIZER}: training loss")
plt.legend()
plt.show()


plt.plot(
    best_history["epoch"],
    best_history["val_auroc"],
    marker="o",
)

plt.xlabel("Epoch")
plt.ylabel("Validation AUROC")
plt.ylim(0, 1)
plt.title(f"{BEST_TOKENIZER}: validation AUROC")
plt.show()

## Confusion matrix

This answers:

> Which class does the best custom model confuse more often?

In [ ]:
cm = confusion_matrix(
    best_predictions["true"],
    best_predictions["predicted"],
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Background", "Binding"],
).plot()

plt.title(
    f"{BEST_TOKENIZER}: confusion matrix"
)
plt.show()

## ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(
    best_predictions["true"],
    best_predictions["probability"],
)

plt.plot(
    fpr,
    tpr,
    label=f"AUROC = {best_row['best_val_auroc']:.3f}",
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"{BEST_TOKENIZER}: ROC curve")
plt.legend()
plt.show()

## Precision–Recall curve

In [ ]:
precision, recall, _ = precision_recall_curve(
    best_predictions["true"],
    best_predictions["probability"],
)

plt.plot(
    recall,
    precision,
    label=f"AUPRC = {best_row['final_val_auprc']:.3f}",
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    f"{BEST_TOKENIZER}: Precision–Recall curve"
)
plt.legend()
plt.show()

## Putting biology and HPC together

For each tokenizer ask two separate questions:

**Scientific**

> How well did it distinguish Binding from Background?

**Computational**

> How long did it take, how much memory did it use, and how many parameters did the model contain?

A useful representation is not automatically the one with the shortest runtime, and the fastest representation is not automatically the best classifier.

# ✅ Group B final questions

1. What stayed the same between Notebook 2 and Notebook 3B?
2. What became larger?
3. Why does each GPU have a DDP rank?
4. Which tokenizer had the best AUROC?
5. Which tokenizer took the longest?
6. What did the confusion matrix reveal that AUROC alone did not?
7. Did the best scientific result also have the lowest computational cost?
8. How might token count help explain runtime differences?

# 🎨 Optional Visualization Playground

This section is **optional**. Nothing below is required to finish the notebook.

Use it when you want to ask your own question about a variable you created earlier.

```mermaid
flowchart LR
    A["Choose a variable"] --> B["Choose a term / column"]
    B --> C{"What do you want to see?"}
    C -->|"Counts / categories"| D["Bar plot"]
    C -->|"Distribution"| E["Histogram"]
    C -->|"Change across epochs"| F["Line plot"]
    C -->|"Relationship between numbers"| G["Scatter plot"]
```

## Two plotting patterns to remember

### One term / column

```python
VARIABLE["TERM"].plot(kind="hist")
```

Read it as:

> From this variable, choose this term, then plot it.

### Two terms / columns

```python
VARIABLE.plot(
    x="TERM1",
    y="TERM2",
    kind="scatter",
)
```

Read it as:

> Use `TERM1` for the x-axis and `TERM2` for the y-axis.

Useful `kind=` choices:

| `kind` | Good for |
|---|---|
| `"bar"` | comparing categories |
| `"hist"` | seeing a distribution |
| `"line"` | following change across epochs |
| `"scatter"` | comparing two numerical values |
| `"box"` | comparing distributions between groups |

If you forget what terms exist inside a pandas table, run:

```python
VARIABLE.columns.tolist()
```

## Important variables from Notebook 3B

| Variable | What it contains | Terms you can explore |
|---|---|---|
| `comparison` | one row for each tokenizer's large multi-GPU run | `tokenizer`, `best_val_auroc`, `final_val_auprc`, `final_val_accuracy`, `training_time_seconds`, `peak_gpu_memory_gb`, `parameters`, plus batch/GPU fields stored in the summaries |
| `best_history` | epoch-by-epoch values for the selected best tokenizer | `epoch`, `train_loss`, `val_auroc`, `val_auprc` |
| `best_predictions_df` | validation predictions for the selected tokenizer | `true`, `predicted`, `probability`, `correct` |
| `best_row` | summary values for the selected best tokenizer | access one value with `best_row["TERM"]` |

### Questions you could visualize

- Which tokenizer performs best?
- Which tokenizer takes longest?
- Which tokenizer uses the most GPU memory?
- Is a larger model/input representation more computationally expensive?
- Did the best tokenizer continue improving over epochs?
- How confident was its classifier?

In [ ]:
# ▶️ OPTIONAL — create a DataFrame for the best model's predictions

best_predictions_df = pd.DataFrame(best_predictions)
best_predictions_df["correct"] = (
    best_predictions_df["true"]
    == best_predictions_df["predicted"]
)

print("comparison terms:")
print(comparison.columns.tolist())

print("\nbest_history terms:")
print(best_history.columns.tolist())

print("\nbest_predictions_df terms:")
print(best_predictions_df.columns.tolist())

## Copy a visualization recipe and change the terms

### Compare biological performance

```python
comparison.plot(
    x="tokenizer",
    y=["best_val_auroc", "final_val_auprc"],
    kind="bar",
)
```

### Compare training time

```python
comparison.plot(
    x="tokenizer",
    y="training_time_seconds",
    kind="bar",
)
```

### Compare GPU memory

```python
comparison.plot(
    x="tokenizer",
    y="peak_gpu_memory_gb",
    kind="bar",
)
```

### Model size vs. training time

```python
comparison.plot(
    x="parameters",
    y="training_time_seconds",
    kind="scatter",
)
```

### Best tokenizer across epochs

```python
best_history.plot(
    x="epoch",
    y=["val_auroc", "val_auprc"],
    kind="line",
    marker="o",
)
```

### Best model's prediction confidence

```python
best_predictions_df["probability"].plot(
    kind="hist",
    bins=20,
)
```

**Student challenge:** find one tradeoff between model quality and computational cost.